In [4]:
!pip install yfinance tqdm arch scikit-learn joblib python-bcb
#!pip install python-bcb

Defaulting to user installation because normal site-packages is not writeable
  Using cached arch-8.0.0-cp314-cp314-win_amd64.whl.metadata (13 kB)
  Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached arch-8.0.0-cp314-cp314-win_amd64.whl (934 kB)
Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl (9.6 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/3 [patsy]
   ---------------------------------------- 0/3 [patsy]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- -------------------------- 1/3 [statsmodels]
   ------------- --------------------------

In [5]:
# Pacotes
!pip install python-bcb

import numpy as np           
import pandas as pd           
import yfinance as yf  
from datetime import date, timedelta
from pathlib import Path
from multiprocessing import cpu_count
from tqdm import tqdm
import warnings
from arch.univariate import ConstantMean, Normal, GARCH, arch_model
from sklearn.metrics import mean_squared_error
import itertools
import random
from joblib import Parallel, delayed
from bcb import sgs
#from utils import ajustar_modelo

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Define o período de tempo para coleta dos dados
# fim    = date.today()
fim    = date(2025, 12, 31)
inicio = fim - timedelta(days=365*2)

# Diretórios
dir_raiz   = "G:\\Meu Drive\\Ensino\\TAF\\taf-code"
input_dir  = Path(dir_raiz, 'Data\\Input\\')
stage_dir  = Path(dir_raiz, 'Data\\Stage\\')
output_dir = Path(dir_raiz, 'Data\\Output\\')
figure_dir = Path(dir_raiz, 'Figures\\')

# Jobs
jobs = cpu_count() - 2

Defaulting to user installation because normal site-packages is not writeable


In [6]:
# Define os ativos a serem utilizados
ativos = ['AGRO3.SA', '^BVSP']  

# Coleta os dados do Yahoo Finance
dados = (yf
         .download(ativos, start=inicio, end=fim, 
                   auto_adjust=False)
         ['Adj Close']
         )
dados.head()

# SELIC (código 432)
selic = (sgs.get({'selic': 432}, 
                start=inicio,
                end = fim)
         .reset_index()
         )

selic.head()

retornos = (dados
            .resample('ME')
            .last()
            .pct_change()
            .dropna()
            )
retornos.head()

log_retornos = (np.log(dados.resample('ME').last()) 
               - np.log(dados.resample('ME').last().shift(1))                
               ).dropna()
log_retornos.head()

# Converter SELIC anual para taxa mensal
selic = (selic
         .assign(selic   = lambda x: x['selic'] / 100,
                 selic_m = lambda x: (1 + x['selic'])**(1/12) - 1))

# Merge dos retornos com a SELIC e cálculo dos excessos
dados = (retornos
         .rename(columns={'^BVSP': 'BVSP'})
         .reset_index()
         .merge(selic[['Date', 'selic_m']], on='Date', how='inner')
         .assign(e_AGRO3 = lambda x: x['AGRO3.SA'] - x['selic_m'],
                 e_BVSP  = lambda x: x['BVSP'] - x['selic_m'])
         )

dados.head()

[*********************100%***********************]  2 of 2 completed


,Date,AGRO3.SA,BVSP,selic_m,e_AGRO3,e_BVSP
0,2024-02-29,-0.038118,0.009925,0.008924,-0.047042,0.001002
1,2024-03-31,0.035413,-0.007084,0.008545,0.026868,-0.015629
2,2024-04-30,0.044788,-0.017033,0.008545,0.036243,-0.025578
3,2024-05-31,-0.010522,-0.030383,0.008355,-0.018877,-0.038739
4,2024-06-30,0.009846,0.014816,0.008355,0.001491,0.006461


In [16]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.rolling import RollingOLS

# Define o modelo CAPM usando a função OLS do statsmodels
modelo = smf.ols('e_AGRO3 ~ e_BVSP', data=dados).fit()

# Imprime os resultados do modelo
print(modelo.summary())

# SELIC (código 432)
selic = (sgs.get({'selic': 432}, 
                start=inicio,
                end = fim)
         .reset_index()
         )

selic.head()

selic = (selic
         .assign(selic   = lambda x: x["selic"]/100,
                 selic_m = lambda x: (1 + x["selic"])**(1/12) - 1,
                 selic_w = lambda x: (1 + x["selic"])**(1/52) - 1,
                 )
         )

selic.head()

# Excessos de retorno
data = (log_retornos
        .merge(selic, how = "inner", 
               left_on = "Date",
               right_on = "Date")
        .rename(columns = {"^BVSP": "BVSP"})
        .rename(columns=lambda x: x.replace('.SA', ''))
        )

# Colunas de ações (excluindo a coluna 'livre_de_risco')
colunas_acoes = [col for col in data.columns 
                 if col not in ['selic', 'selic_m', 'selic_w', 'Date'] ]

data = (data
        .assign(**{f'e_{col}': lambda x, c=col: x[c] - x['selic_w'] 
                   for col in colunas_acoes })
        .rename(columns=lambda x: x.replace('.SA', ''))
        )

data.head()

# Rolling CAPM - janela de 30 semanas
window = 30
results = []
acoes_r = [c for c in colunas_acoes if c != 'BVSP']
for acao in acoes_r:
    y = data[f'e_{acao}']
    X = sm.add_constant(data['e_BVSP'])
    
    rolling = RollingOLS(y, X, window=window).fit()
    
    params  = np.array(rolling.params)    
    pvalues = np.array(rolling.pvalues)   
    ci = rolling.conf_int()
    
    n = len(params)
    for i in range(window - 1, n):
        results.append({
            'Ação': acao,
            'Data_Fim': data['Date'].iloc[i],
            'Beta':   params[i, 1],
            'Alfa':   params[i, 0],
            'P_Beta': pvalues[i, 1],
            'P_Alfa': pvalues[i, 0],
            'Lower_Alfa': ci[('const',   'lower')].values[i],
            'Upper_Alfa': ci[('const',   'upper')].values[i],
            'Lower_Beta': ci[('e_BVSP',  'lower')].values[i],
            'Upper_Beta': ci[('e_BVSP',  'upper')].values[i]
        })

resultados_rolling = (pd.DataFrame(results)
                       .sort_values(['Ação', 'Data_Fim'])
                       .reset_index(drop=True))

resultados_rolling.head()


                            OLS Regression Results                            
Dep. Variable:                e_AGRO3   R-squared:                       0.057
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     1.260
Date:                dom, 28 jun 2026   Prob (F-statistic):              0.274
Time:                        20:31:08   Log-Likelihood:                 41.721
No. Observations:                  23   AIC:                            -79.44
Df Residuals:                      21   BIC:                            -77.17
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0142      0.009     -1.650      0.1

IndexError: index 29 is out of bounds for axis 0 with size 23

In [12]:
import matplotlib.pyplot as plt

# Criando os gráficos
for acao in resultados_rolling['Ação'].unique():
    plt.close()
    df_acao = resultados_rolling[resultados_rolling['Ação'] == acao]

    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), sharex=True)

    # Gráfico do Beta
    axes[0].plot(df_acao['Data_Fim'], df_acao['Beta'], label='Beta', color='red')
    axes[0].plot(df_acao['Data_Fim'], df_acao['Lower_Beta'], label='Lower Beta', color='grey')
    axes[0].plot(df_acao['Data_Fim'], df_acao['Upper_Beta'], label='Upper Beta', color='grey')
    axes[0].set_ylabel('Beta')
    axes[0].set_title(f'Evolução dos Parâmetros do CAPM - Ação {acao}')
    axes[0].legend()
    axes[0].grid(True)

    # Gráfico do Alfa
    axes[1].plot(df_acao['Data_Fim'], df_acao['Alfa'], label='Alfa', color='orange')
    axes[1].plot(df_acao['Data_Fim'], df_acao['Lower_Alfa'], label='Lower Alfa', color='grey')
    axes[1].plot(df_acao['Data_Fim'], df_acao['Upper_Alfa'], label='Upper Alfa', color='grey')
    axes[1].set_ylabel('Alfa')
    axes[1].legend()
    axes[1].grid(True)
    
    fig.get_figure().savefig( Path(figure_dir, 'capm_weekly_rolling_' + acao + '.png'), dpi = 400)
    plt.tight_layout()

plt.show()

NameError: name 'resultados_rolling' is not defined